# M4.1: chunking strategy evaluation

This notebook compares Fixed, Recursive, and the production Structure-Aware Chunker. The dataset, embedding model, query encoding, cosine retrieval, and metrics are fixed; chunking strategy is the only intended variable. Ground truth uses source block IDs, never generated chunk IDs.

In [1]:
# Run in a fresh Google Colab runtime. Replace the repository URL if using a fork.
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = 'https://github.com/ozgemelteminan/prompt-generator-rag'  # Replace with your repository URL.
repository = Path('prompt-generator-rag')
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL], check=True)
os.chdir(repository)
subprocess.run(['pip', 'install', '-q', '-e', 'packages/prompt-engine'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'apps/api', '--no-deps', 'sentence-transformers'], check=True)

CompletedProcess(args=['pip', 'install', '-q', '-e', 'apps/api', '--no-deps', 'sentence-transformers'], returncode=0)

In [7]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/prompt-generator-rag")
API_ROOT = REPO_ROOT / "apps" / "api"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))

print("Repo root:", REPO_ROOT)
print("API root:", API_ROOT)
print("API import path configured.")

Repo root: /content/prompt-generator-rag
API root: /content/prompt-generator-rag/apps/api
API import path configured.


In [9]:
from app.document_processing.chunking import StructureAwareChunker, Tokenizer
from app.document_processing.models import ChunkingConfig, TextBlock

print("✅ Production chunker imported successfully")

✅ Production chunker imported successfully


In [13]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "Alibaba-NLP/gte-multilingual-base"

model = SentenceTransformer(
    MODEL_NAME,
    trust_remote_code=True
)

print("✅ Model loaded")

configuration.py:   0%|          | 0.00/7.13k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/new-impl:
- configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling.py:   0%|          | 0.00/59.0k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/new-impl:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors: reconstructing file:   0%|          |  0.00B /  611MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded


In [15]:
production_config = ChunkingConfig(target_tokens=350, max_tokens=500, overlap_tokens=40)
strategies = {
    'fixed': tuple(chunk for document in dataset.documents for chunk in fixed_size_chunks(document, max_tokens=500, overlap_tokens=40)),
    'recursive': tuple(chunk for document in dataset.documents for chunk in recursive_chunks(document, target_tokens=350, max_tokens=500)),
    # Calls apps/api/app/document_processing/chunking.py; it is not copied here.
    'production_structure_aware': tuple(chunk for document in dataset.documents for chunk in production_structure_aware_chunks(document, config=production_config)),
}
configurations = {
    'fixed': {'max_tokens': 500, 'overlap_tokens': 40},
    'recursive': {'target_tokens': 350, 'max_tokens': 500},
    'production_structure_aware': {'target_tokens': 350, 'max_tokens': 500, 'overlap_tokens': 40},
}
results = run_comparison(dataset, embedder=embedder, strategies=strategies)

NameError: name 'embedder' is not defined

In [ ]:
import pandas as pd

comparison = pd.DataFrame([{'Chunker': result.name, 'Avg Tokens': result.chunk_statistics['mean_tokens'], 'Recall@5': result.overall['recall_at_5'], 'Recall@10': result.overall['recall_at_10'], 'MRR': result.overall['mrr'], 'nDCG@10': result.overall['ndcg_at_10'], 'HitRate@5': result.overall['hit_rate_at_5']} for result in results])
display(comparison)
for metric, label in [('Recall@10', 'best Recall@10'), ('MRR', 'best MRR'), ('nDCG@10', 'best nDCG@10')]:
    winners = comparison.loc[comparison[metric] == comparison[metric].max(), 'Chunker'].tolist()
    print(f'{label}: {winners}')
for result in results:
    print(f'\n{result.name}: category breakdown')
    display(pd.DataFrame(result.by_category).T)
    print(f'{result.name}: language breakdown')
    display(pd.DataFrame(result.by_language).T)

save_results(results, dataset_version=dataset.version, embedder=embedder, output_dir=ROOT / 'evals/results/chunking', chunker_configurations=configurations)

## Reporting rule

Report the best Recall@10, MRR, and nDCG@10 independently. Recommend a strategy only when it wins the quality metric relevant to the use case without an unacceptable chunk-size or truncation tradeoff; otherwise report the tradeoff. Do not create a composite score.